# Section D: Integration, conclusions and outputs

**Everyone:** Ronah Abucheli, Phillip Leo Jnr, Njihia Muranga, Gichuru Samuel, Mwaniki Kelvin Maina and Gakuo Philip Ndirangu

How conserved is the Ebola genome across *Ebolavirus* species, and can genome sequence alone separate Bundibugyo from Zaire and Sudan?

This notebook covers stages 7 and 8 of the project.

* **Needs:** `handoff_A.pkl`, `handoff_B.pkl` and `handoff_C.pkl` in `ebola_project/outputs/`
* **Saves:** the integration table and figure, draft conclusions, `README.md`, `results_summary.json`, and the two results zip files

The four section notebooks are run in order: A, then B and C (either order), then D. Run them in the same Colab session, or upload the earlier `handoff_*.pkl` files into `ebola_project/outputs/` first. Keep the settings cell the same in all four notebooks, with your email in `NCBI_EMAIL`.

## Setup

Installs the tools the notebook needs (Biopython, MAFFT, FastTree) and loads the Python libraries.

In [ ]:
import os, sys, re, io, json, time, gzip, shutil, subprocess, pickle, hashlib, zipfile, warnings, random, contextlib, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sys.setrecursionlimit(20000)

IN_COLAB = "google.colab" in sys.modules

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr[-2000:]}")
    return r

try:
    import Bio
except ImportError:
    sh("pip -q install biopython")
    import Bio
if not (shutil.which("mafft") and (shutil.which("fasttree") or shutil.which("FastTree"))):
    print("Installing MAFFT and FastTree (about 1 minute)...")
    sh("apt-get -qq update && apt-get -qq install -y mafft fasttree", check=False)
print("MAFFT:", shutil.which("mafft"), "| FastTree:", shutil.which("fasttree") or shutil.which("FastTree"))
print("CPU cores:", os.cpu_count(), "| Biopython", Bio.__version__)

The settings for the whole run. Put your own email in `NCBI_EMAIL` (NCBI asks for it when downloading). The other settings can stay as they are.

In [ ]:
NCBI_EMAIL = "your_email@example.com"  # put your own email here
NCBI_API_KEY = None
DOWNLOAD_METHOD = "datasets"
ENTREZ_MAX_PER_SPECIES = 2500

MIN_LEN, MAX_LEN = 17000, 20500
MAX_N_FRACTION = 0.01  # drop genomes with more than 1% unknown bases
ZAIRE_CAP_PER_OUTBREAK = 40  # keep at most this many Zaire genomes per outbreak
RANDOM_SEED = 42

EDA_KMER_K = 4
KMER_K = 6  # k-mer length used by the classifiers
ML_CLASSES = ["Bundibugyo", "Sudan", "Zaire"]
FRAGMENT_TEST = True
FRAGMENT_LENGTHS = [250, 500, 1000, 2500]
N_JOBS = -1

ALIGN_MAX_SEQS = 400  # align at most this many genomes
MAFFT_MODE = "fast"  # "fast" or "auto" (slower, more accurate)
TREE_GAMMA = False
REUSE_CACHE = True

WORKDIR = Path("ebola_project")
DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR = [WORKDIR / d for d in ("data", "data/refs", "figures", "outputs")]
for d in (DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

SPECIES = ["Zaire", "Sudan", "Bundibugyo", "Reston", "Tai Forest", "Bombali"]
SPECIES_QUERY = {
    "Zaire": ["Orthoebolavirus zairense", "Zaire ebolavirus"],
    "Sudan": ["Orthoebolavirus sudanense", "Sudan ebolavirus"],
    "Bundibugyo": ["Orthoebolavirus bundibugyoense", "Bundibugyo ebolavirus"],
    "Reston": ["Orthoebolavirus restonense", "Reston ebolavirus"],
    "Tai Forest": ["Orthoebolavirus taiense", "Tai Forest ebolavirus"],
    "Bombali": ["Orthoebolavirus bombaliense", "Bombali ebolavirus"],
}
REF_ACC = {"Zaire": "NC_002549.1", "Sudan": "NC_006432.1", "Bundibugyo": "NC_014373.1",
           "Reston": "NC_004161.1", "Tai Forest": "NC_014372.1", "Bombali": "NC_039345.1"}
GENE_ORDER = ["NP", "VP35", "VP40", "GP", "VP30", "VP24", "L"]
COLORS = {"Zaire": "#1f77b4", "Sudan": "#d62728", "Bundibugyo": "#2ca02c",
          "Reston": "#9467bd", "Tai Forest": "#ff7f0e", "Bombali": "#8c564b"}

TIMINGS, RESULTS = [], {}
@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"> {label} ...")
    try: yield
    finally:
        dt = time.time() - t; TIMINGS.append((label, dt)); print(f"< {label}: {dt/60:.1f} min ({dt:.0f} s)")

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

if "example.com" in NCBI_EMAIL:
    raise ValueError("Set NCBI_EMAIL (above) to your real email address before running the real download.")

Helper functions for reading and writing FASTA files, finding the genes in a reference genome and downloading data from NCBI.

In [ ]:
def read_fasta(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as fh:
        name, desc, buf = None, "", []
        for line in fh:
            if line.startswith(">"):
                if name is not None: yield name, desc, "".join(buf).upper()
                head = line[1:].strip().split(None, 1)
                name, desc, buf = head[0], (head[1] if len(head) > 1 else ""), []
            else: buf.append(line.strip())
        if name is not None: yield name, desc, "".join(buf).upper()

def write_fasta(path, items, width=80):
    with open(path, "w") as fh:
        for name, seq in items:
            fh.write(f">{name}\n")
            for i in range(0, len(seq), width): fh.write(seq[i:i+width] + "\n")

def base_acc(a): return a.split(".")[0]

def canon_gene(q):
    g = (q.get("gene", [""])[0]).strip(); prod = (q.get("product", [""])[0]).lower()
    if g.upper() in GENE_ORDER: return g.upper()
    if "nucleoprotein" in prod: return "NP"
    if "vp35" in prod or "cofactor" in prod: return "VP35"
    if "vp40" in prod or "matrix" in prod: return "VP40"
    if "vp30" in prod: return "VP30"
    if "vp24" in prod: return "VP24"
    if "glycoprotein" in prod and "secreted" not in prod and "small" not in prod: return "GP"
    if "polymerase" in prod or prod.strip() == "l": return "L"
    return None

def gene_table(rec):
    spans = {}
    for f in rec.features:
        if f.type != "CDS": continue
        g = canon_gene(f.qualifiers)
        if g is None: continue
        s, e = int(f.location.start) + 1, int(f.location.end)
        a, b = spans.get(g, (s, e)); spans[g] = (min(a, s), max(b, e))
    df = pd.DataFrame([(g, *spans[g]) for g in GENE_ORDER if g in spans], columns=["gene", "start", "end"])
    missing = [g for g in GENE_ORDER if g not in spans]
    if missing: print(f"  WARNING {rec.id}: genes not found in annotation: {missing}")
    return df

def fetch_refs():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    refs, genes = {}, {}
    for sp, acc in REF_ACC.items():
        p = REF_DIR / f"{base_acc(acc)}.gb"
        if not p.exists():
            with Entrez.efetch(db="nuccore", id=base_acc(acc), rettype="gb", retmode="text") as h: p.write_text(h.read())
            time.sleep(0.4)
        rec = SeqIO.read(p, "genbank")
        refs[sp] = str(rec.seq).upper(); genes[sp] = gene_table(rec)
        REF_ACC[sp] = rec.id
    return refs, genes

def jget(d, *keys):
    for k in keys:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def download_datasets_cli():
    ds = DATA_DIR / "datasets"
    if not ds.exists():
        sh(f"curl -sSL -o {ds} https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets && chmod +x {ds}")
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        got = False
        for nm in names:
            z = DATA_DIR / f"{sp.replace(' ', '_')}.zip"
            r = sh(f'{ds} download virus genome taxon "{nm}" --complete-only --filename {z}', check=False)
            if r.returncode != 0 or not z.exists(): continue
            out = DATA_DIR / f"unz_{sp.replace(' ', '_')}"; shutil.rmtree(out, ignore_errors=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
            fna = next(out.rglob("genomic.fna"), None); rep = next(out.rglob("data_report.jsonl"), None)
            if fna is None: continue
            seqs = {n: (d, s) for n, d, s in read_fasta(fna)}
            reports = {}
            if rep is not None:
                for line in open(rep):
                    j = json.loads(line); reports[j.get("accession")] = j
            for acc, (desc, s) in seqs.items():
                j = reports.get(acc, {})
                rows.append(dict(accession=acc, species=sp, isolate=jget(j, "isolate", "name") or desc,
                                 collection_date=jget(j, "isolate", "collectionDate"),
                                 geo=jget(j, "location", "geographicLocation"), region=jget(j, "location", "geographicRegion"),
                                 desc=desc))
                records[acc] = s
            print(f"  {sp}: {len(seqs)} genomes (query '{nm}')"); got = True; break
        if not got: print(f"  WARNING: nothing downloaded for {sp}")
    return records, pd.DataFrame(rows)

def download_entrez():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        for nm in names:
            term = f'"{nm}"[Organism] AND {MIN_LEN}:{MAX_LEN}[SLEN]'
            res = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=0, usehistory="y"))
            n = int(res["Count"])
            if n > 0: break
        n = min(n, ENTREZ_MAX_PER_SPECIES); print(f"  {sp}: fetching {n} records")
        for start in range(0, n, 200):
            for attempt in range(4):
                try:
                    h = Entrez.efetch(db="nuccore", rettype="gb", retmode="text", retstart=start, retmax=min(200, n - start),
                                      webenv=res["WebEnv"], query_key=res["QueryKey"])
                    batch = list(SeqIO.parse(h, "genbank")); break
                except Exception as ex:
                    print("   retry", attempt, ex); time.sleep(3); batch = []
            for rec in batch:
                src = next((f for f in rec.features if f.type == "source"), None); q = src.qualifiers if src else {}
                records[rec.id] = str(rec.seq).upper()
                rows.append(dict(accession=rec.id, species=sp, isolate=(q.get("isolate") or q.get("strain") or [""])[0],
                                 collection_date=(q.get("collection_date") or [None])[0], geo=(q.get("country") or [None])[0],
                                 region=None, desc=rec.description))
            time.sleep(0.4)
    return records, pd.DataFrame(rows)

Loads the results saved by Sections A, B and C.

In [ ]:
_hA = pickle.load(open(OUT_DIR / "handoff_A.pkl", "rb")); _hB = pickle.load(open(OUT_DIR / "handoff_B.pkl", "rb")); _hC = pickle.load(open(OUT_DIR / "handoff_C.pkl", "rb"))
qc, SEQ, REFS, GENES = _hA["qc"], _hA["SEQ"], _hA["REFS"], _hA["GENES"]; REF_ACC.update(_hA["REF_ACC"])
gene_cons, cons_sp_r, diag, mono, BUND_SISTER = _hB["gene_cons"], _hB["cons_sp_r"], _hB["diag"], _hB["mono"], _hB["BUND_SISTER"]
ranked, cod, sp_present, n_seq = _hB["ranked"], _hB["cod"], _hB["sp_present"], _hB["n_seq"]
results, piv, PROF = _hC["results"], _hC["piv"], _hC["PROF"]
for _h in (_hA, _hB, _hC): RESULTS.update(_h["RESULTS"]); TIMINGS.extend(_h["TIMINGS"])
gt = GENES["Zaire"]
win = lambda x, w=150: pd.Series(x).rolling(w, center=True, min_periods=1).mean().values
print(f"Loaded: {len(qc)} genomes, {n_seq} aligned, {len(results)} model results")

## 7. Integration and conclusions

Brings the results together.

Puts conservation, diagnostic sites and classifier importance side by side for each gene, and checks whether they follow the same pattern along the genome.

In [ ]:
gdiag = diag[(diag.comparison == "Bundibugyo vs Zaire+Sudan") & (diag.criterion == ">=95% within class")].iloc[0]
integ = pd.DataFrame({
    "gene": GENE_ORDER,
    "length_nt": [int(gt.set_index("gene").loc[g].end - gt.set_index("gene").loc[g].start + 1) for g in GENE_ORDER],
    "cons_across_species": [gene_cons.set_index("gene").loc[g].cons_across_species for g in GENE_ORDER],
    "bund_diagnostic_sites_95": [int(gdiag[g]) for g in GENE_ORDER],
    "classifier_enrichment_Zaire_ref": [piv.loc[g, "Zaire"] for g in GENE_ORDER]})
integ["diagnostic_sites_per_kb"] = (integ.bund_diagnostic_sites_95 / integ.length_nt * 1000).round(1)
display(integ.round(3)); integ.to_csv(OUT_DIR / "integration_by_gene.csv", index=False)

from scipy.stats import spearmanr
bins = np.arange(0, len(REFS["Zaire"]), 200)
bc = [np.nanmean(cons_sp_r[b:b + 200]) for b in bins]; bi = [PROF["Zaire"][b:b + 200].mean() for b in bins]
rho, pval = spearmanr(bc, bi, nan_policy="omit"); RESULTS["rho_cons_vs_importance"] = [float(rho), float(pval)]
print(f"Spearman correlation between across-species conservation and classifier importance (200-nt bins): rho = {rho:.2f}, p = {pval:.2g}")

fig, ax = plt.subplots(figsize=(16, 4.5)); ax2 = ax.twinx()
ax.plot(win(cons_sp_r), color="black", lw=1.1, label="across-species conservation"); ax2.plot(win(PROF["Zaire"], 300), color="#d62728", lw=1.1, label="classifier importance")
for i, g in enumerate(gt.itertuples()): ax.axvspan(g.start, g.end, color="#dddddd" if i % 2 == 0 else "#bbbbbb", alpha=.35, lw=0); ax.text((g.start + g.end) / 2, ax.get_ylim()[1], g.gene, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("conservation"); ax2.set_ylabel("k-mer importance"); ax.set_xlabel("Zaire reference position (nt)"); ax.set_title("Conservation vs. classifier importance along the genome", pad=18)
ax.legend(loc="lower left"); ax2.legend(loc="lower right"); savefig("07_conservation_vs_importance")

Writes draft conclusions from the numbers computed above. Edit the wording in your own words.

In [ ]:
rf_l = results[(results.model == "Random forest") & (results.validation == "Leave-one-outbreak-out")].iloc[0]
lr_l = results[(results.model == "Logistic regression") & (results.validation == "Leave-one-outbreak-out")].iloc[0]
bl_l = results[(results.model == "Majority baseline") & (results.validation == "Leave-one-outbreak-out")].iloc[0]
cv_rows = results[results.validation.str.startswith("Stratified")]
best = max([rf_l, lr_l], key=lambda r: r.balanced_accuracy)
sep = "does" if (best.recall_Bundibugyo >= 0.95 and best.balanced_accuracy >= 0.95) else "only partly"
lines = [
 f"1. Conservation. Across the {len(sp_present)} species, {RESULTS['genome_invariant_across_species_pct']:.1f}% of Zaire-reference positions are identical in every species' consensus. "
 f"{ranked.iloc[0].gene} is the most conserved gene ({ranked.iloc[0].cons_across_species:.3f}) and {ranked.iloc[-1].gene} the least ({ranked.iloc[-1].cons_across_species:.3f}).",
 f"2. Phylogeny. " + "; ".join(f"{r.species} {'is a single clade' if r.monophyletic else 'is not monophyletic'}" for r in mono.itertuples()) +
 (f". Bundibugyo's sister group is {', '.join(BUND_SISTER.index[:2])}." if BUND_SISTER is not None and len(BUND_SISTER) else "."),
 f"3. Diagnostic sites. {RESULTS['diag_bund_fixed']} alignment positions are fixed within each of Bundibugyo, Sudan and Zaire while Bundibugyo differs from both "
 f"({RESULTS['diag_bund_95']} at the >=95% within-class criterion).",
 f"4. Classification. Under leave-one-outbreak-out validation the majority baseline scores {bl_l.accuracy:.1%} accuracy, logistic regression {lr_l.accuracy:.1%} "
 f"(balanced {lr_l.balanced_accuracy:.1%}) and random forest {rf_l.accuracy:.1%} (balanced {rf_l.balanced_accuracy:.1%}). Bundibugyo recall: "
 f"logistic regression {lr_l.recall_Bundibugyo:.1%}, random forest {rf_l.recall_Bundibugyo:.1%}. Genome sequence alone {sep} separate Bundibugyo from Zaire and Sudan in this dataset.",
]
if len(cv_rows): lines.append("5. A random stratified split gives " + ", ".join(f"{r.model} {r.accuracy:.1%}" for r in cv_rows.itertuples()) + " accuracy, to be compared with the leave-one-outbreak-out figures above.")
if "fragments" in RESULTS:
    f = pd.DataFrame(RESULTS["fragments"]); lines.append("6. Fragments. Balanced accuracy is " + ", ".join(f"{r.balanced_accuracy:.0%} at {int(r.fragment_nt)} nt" for r in f.itertuples()) + ".")
lines.append(f"7. Link between conservation and the classifier. Spearman rho between conservation and classifier importance across the genome is {rho:.2f}.")
print("\n".join(lines)); RESULTS["conclusion_draft"] = lines

### Limitations

* **Imbalance.** Bundibugyo has far fewer genomes than Zaire, and they come from only a few outbreaks. Class weights and per-species recall help but do not remove this.
* **Sampling bias.** GenBank has more genomes from well-studied outbreaks. Capping Zaire per outbreak reduces this but does not remove it.
* **Outbreak labels are inferred** from country and collection year, so a wrong label weakens the "unseen outbreak" test.
* **Related genomes.** Genomes from the same outbreak are nearly identical, so the real number of independent samples is smaller than the genome count.
* **Species labels** come from NCBI taxonomy.
* **Speed settings.** Aligning a subset with fast MAFFT, and building the tree with FastTree, trade some accuracy for speed.
* **Species are easy to tell apart.** The species differ by 30% or more, so high accuracy is expected. This is why the models are tested on unseen outbreaks, and why feature importance and short fragments are also examined.

## 8. Outputs

Saves the results so the work can be repeated and shared.

Saves a summary of the results as a JSON file and writes a README for the project.

In [ ]:
RESULTS["timings_min"] = {k: round(v / 60, 2) for k, v in TIMINGS}
json.dump(RESULTS, open(OUT_DIR / "results_summary.json", "w"), indent=2, default=str)
readme = f"""# Conservation of the Ebola genome across *Ebolavirus* species

## Question
How conserved is the Ebola genome across *Ebolavirus* species, and can genome sequence alone separate Bundibugyo from Zaire and Sudan?

## Repository layout
- `ebola_conservation_project.ipynb`: the full analysis, eight stages (run top to bottom, Google Colab compatible)
- `ebola_project/data/`: downloaded NCBI genomes, metadata, reference GenBank files
- `ebola_project/outputs/`: QC set, alignment, tree, result tables (`results_summary.json`)
- `ebola_project/figures/`: all figures produced by the notebook

## Data
Complete *Ebolavirus* genomes from NCBI Virus / NCBI Datasets, six RefSeq references ({', '.join(REF_ACC.values())}).
Downloaded: {RESULTS['n_downloaded']}; after QC and the Zaire cap ({ZAIRE_CAP_PER_OUTBREAK} per outbreak): {RESULTS['n_qc']}.
Per species: {RESULTS['qc_by_species']}.

## Methods
1. QC: length {MIN_LEN}-{MAX_LEN} nt, <= {MAX_N_FRACTION:.0%} ambiguous bases, exact duplicates removed, outbreak derived from country and year.
2. Exploratory: genome lengths, counts, PCA of {EDA_KMER_K}-mer profiles.
3. MAFFT alignment ({MAFFT_MODE} mode, {n_seq} sequences), per-position and per-gene conservation, FastTree GTR tree (midpoint rooted).
4. Classification of Bundibugyo / Sudan / Zaire from {KMER_K}-mer frequencies: majority baseline, logistic regression, random forest; leave-one-outbreak-out validation; feature importance mapped to genes; short-fragment test.

## Key results
""" + "\n".join(f"- {l}" for l in RESULTS["conclusion_draft"]) + f"""

## Reproducing
Open the notebook in Colab and run all cells. Total run time on the original run: {sum(v for _, v in TIMINGS)/60:.0f} minutes.
Random seed: {RANDOM_SEED}. Configuration is in the first code cell.
"""
(WORKDIR / "README.md").write_text(readme); print(readme[:1500], "...")

Prints how long each step took and saves two zip files: the full results, and a smaller pack of figures and tables.

In [ ]:
tt = pd.DataFrame(TIMINGS, columns=["step", "seconds"]); tt["minutes"] = (tt.seconds / 60).round(2)
print(tt.sort_values("seconds", ascending=False).to_string(index=False)); print(f"\nTOTAL: {tt.seconds.sum()/60:.1f} minutes")
shutil.make_archive("ebola_project_results", "zip", WORKDIR); print("Results archive: ebola_project_results.zip")
if IN_COLAB:
    from google.colab import files; files.download("ebola_project_results.zip")

with zipfile.ZipFile("slides_pack.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIG_DIR.glob("*.png")): zf.write(p, f"figures/{p.name}")
    for p in sorted(OUT_DIR.glob("*.csv")): zf.write(p, f"outputs/{p.name}")
    for nm in ("results_summary.json", "tree.nwk"): zf.write(OUT_DIR / nm, f"outputs/{nm}")
    zf.write(WORKDIR / "README.md", "README.md")
print("Slides pack: slides_pack.zip (%.1f MB)" % (Path("slides_pack.zip").stat().st_size / 1e6))
if IN_COLAB:
    files.download("slides_pack.zip")

## Next steps

* Check the outbreak table from Stage 2 against the downloaded data.
* Read the draft conclusions and put them in your own words.
* The figures are saved in `ebola_project/figures/`.